# Data Quality Findings — Online Retail II

Initial inspection of 1,067,371 transaction rows (Dec 2009 – Dec 2011) revealed the following issues to address during cleaning:

- **Missing Customer ID**: 243,007 rows (~22.8%). Likely guest/non-account transactions.
  Decision needed: exclude from customer-level analysis (RFM, repeat purchases) while retaining for overall revenue figures.
- **Missing Description**: 4,382 rows (~0.4%). Minor — likely tied to unusual StockCode entries.
- **Duplicate rows**: 34,335 exact duplicates (~3.2%). Needs investigation before deciding whether to drop.
- **Negative Quantity**: minimum of -80,995 — far beyond a typical cancellation, worth isolating as an outlier.
- **Negative Price**: minimum of -53,594.36 — not a valid transaction price, likely a data artifact.
- **Non-country Country values**: e.g. "European Community" (61 rows) — needs a decision on reclassification or exclusion.
- **Cancelled transactions**: 19,494 rows (~1.8%) — Invoice values starting with "C". Will be excluded from revenue calculations.

These will be addressed in Day 2 (cleaning).

In [19]:
import pandas as pd
import numpy as np

sheet_2009 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2009-2010")
sheet_2010 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2010-2011")

df = pd.concat([sheet_2009, sheet_2010], ignore_index=True)

df.shape

(1067371, 8)

In [20]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [21]:
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [23]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [24]:
df.duplicated().sum()

np.int64(34335)

In [25]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [26]:
df["Country"].value_counts()

Country
United Kingdom          981330
EIRE                     17866
Germany                  17624
France                   14330
Netherlands               5140
Spain                     3811
Switzerland               3189
Belgium                   3123
Portugal                  2620
Australia                 1913
Channel Islands           1664
Italy                     1534
Norway                    1455
Sweden                    1364
Cyprus                    1176
Finland                   1049
Austria                    938
Denmark                    817
Unspecified                756
Greece                     663
Japan                      582
USA                        535
Poland                     535
United Arab Emirates       500
Israel                     371
Hong Kong                  364
Singapore                  346
Malta                      299
Iceland                    253
Canada                     228
Lithuania                  189
RSA                        169


In [27]:
df["Invoice"] = df["Invoice"].astype(str)
df["Invoice"].str.startswith("C").sum()

np.int64(19494)

In [28]:
df[df["Quantity"] < -1000]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.00,NaN,United Kingdom
7205,490016,21982,NaN,-1012,2009-12-03 12:30:00,0.00,NaN,United Kingdom
26005,491639,20668,NaN,-1395,2009-12-11 16:02:00,0.00,NaN,United Kingdom
47635,493821,17129D,missing (wrongly coded?),-2127,2010-01-07 12:43:00,0.00,NaN,United Kingdom
65138,495199,17061,NaN,-1198,2010-01-21 16:25:00,0.00,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
970589,574822,85036B,damages wax,-1284,2011-11-07 10:38:00,0.00,NaN,United Kingdom
998302,576764,16008,check,-1510,2011-11-16 13:10:00,0.00,NaN,United Kingdom
1040196,579742,85204,lost??,-1131,2011-11-30 14:34:00,0.00,NaN,United Kingdom
1060796,581212,22578,lost,-1050,2011-12-07 18:38:00,0.00,NaN,United Kingdom


In [29]:
df[df["Quantity"] < -10000]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


In [30]:
df[(df["Quantity"] < 0) & (df["Price"] == 0)]["Description"].value_counts()

Description
check              123
damages             84
?                   83
damaged             78
missing             27
                  ... 
wet?                 1
lost??               1
???                  1
wet boxes            1
????damages????      1
Name: count, Length: 222, dtype: int64

In [31]:
df["is_stock_adjustment"] = (df["Quantity"] < 0) & (df["Price"] == 0)

In [32]:
df["Invoice"] = df["Invoice"].astype(str)
df["is_cancelled"] = df["Invoice"].str.startswith("C")

In [33]:
non_product_codes = ["POST", "D", "DOT", "M", "MANUAL", "BANK CHARGES", "CRUK"]
df["is_non_product"] = df["StockCode"].isin(non_product_codes)

In [34]:
before = len(df)
df = df[df["Price"] >= 0]
print(f"Removed {before - len(df)} rows with negative price")

Removed 5 rows with negative price


In [35]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} exact duplicate rows")

Removed 34335 exact duplicate rows


In [36]:
df["Description"] = df["Description"].fillna("Unknown")

In [37]:
df_with_customer = df[df["Customer ID"].notna()].copy()
print(f"{len(df_with_customer)} rows have a Customer ID, out of {len(df)} total")

797885 rows have a Customer ID, out of 1033031 total


In [38]:
df["Revenue"] = df["Quantity"] * df["Price"]

In [39]:
df_revenue_ready = df[
    (~df["is_cancelled"]) &
    (~df["is_non_product"]) &
    (~df["is_stock_adjustment"])
].copy()

print(f"{len(df_revenue_ready)} rows remain for revenue analysis, out of {len(df)} total")
print(f"Total clean revenue: £{df_revenue_ready['Revenue'].sum():,.2f}")

1006362 rows remain for revenue analysis, out of 1033031 total
Total clean revenue: £19,700,580.55


In [40]:
df.to_csv("../data/processed/cleaned_online_retail.csv", index=False)
df_revenue_ready.to_csv("../data/processed/revenue_ready.csv", index=False)

# Day 2 — Cleaning Decisions

- **Cancellations** (Invoice starting with "C"): flagged via `is_cancelled`, not deleted —
  needed intact for cancellation-analysis questions later.
- **Extreme negative Quantity (-80,995)**: investigated individually. Found to be a genuine
  cancellation (Invoice C581484, real Customer ID and Price) — no special handling needed
  beyond the existing `is_cancelled` flag.
- **Stock adjustments**: identified 222 distinct free-text descriptions (e.g. "check", "damaged",
  "lost", "wet boxes") on rows with negative Quantity and Price = 0. These represent internal
  inventory write-offs, not customer transactions. Flagged via `is_stock_adjustment`.
- **Negative Price**: 5 rows removed outright — genuine data artifacts with no analytical value.
- **Duplicate rows**: sampled and confirmed as [genuine repeated line items / data entry errors —
  fill in what you actually found]. 34335 duplicates removed.
- **Missing Description**: filled with "Unknown" — rows retained since they're still real
  transactions.
- **Missing Customer ID**: NOT filled or dropped from the main dataset. Instead, a separate
  `df_with_customer` view was created for customer-level analysis (RFM, repeat purchases),
  while the main `df` retains all rows for accurate revenue totals.
- **Revenue** calculated as Quantity × Price.
- Three output datasets produced: `df` (all rows, flagged), `df_with_customer` (has a real
  customer), `df_revenue_ready` (excludes cancellations, non-products, and stock adjustments) —
  each suited to different upcoming analysis questions.